In [237]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.feature_extraction.text import CountVectorizer

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [238]:
# T1,T2
task1 = test[(test['AwayTeam'] == 'Chelsea') | (test['HomeTeam'] == 'Chelsea')].shape[0]

aggressiveStyles = ['AggressiveTackler', 'RiskTaker', 'HighPressure', 'ChaosInducer']
def calcSAS(styles):
    if pd.isna(styles):
        return 0
    styles_list = [style.strip() for style in str(styles).split(',')]
    aggCount = sum(1 for s in styles_list if s in aggressiveStyles)

    return round((aggCount / len(styles_list)), 2)

test['SAS'] = test['TeamStyles'].apply(calcSAS)
train['SAS'] = train['TeamStyles'].apply(calcSAS)

In [232]:
# T3
y_train = train['chaos_label']
x_train = train.drop(columns=['MatchID', 'chaos_label'])
x_test = test.drop(columns=['MatchID'])

targetCols = ['Season', 'HomeTeam', 'AwayTeam']
oneHotCols = ['FullTimeResult']

numCols = x_train.select_dtypes(exclude='object').columns
# Helper function for splitting teamstyles into a list
def splitString(text):
    if pd.isna(text):
        return []
    return [item.strip() for item in str(text).strip(',')]


preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'), oneHotCols),
        ('target', TargetEncoder(), targetCols),
        ('scale', StandardScaler(), numCols),
        # TeamStyles is a list so encode separately
        ('list', CountVectorizer(tokenizer=splitString, token_pattern=None, binary=True), 'TeamStyles')
    ]
)

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5],
    'model__learning_rate': [0.05, 0.1]
}

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', XGBClassifier(random_state=42))
])

gs = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5
)

gs.fit(x_train, y_train)
model = gs.best_estimator_
predictions = model.predict(x_test)

In [257]:
# DataFrame
rows =[]
test_indexed = test.set_index('MatchID') # For line 14

rows.append({
    'subtaskID': 1,
    'datapointID': 1,
    'answer': task1
})
for id,pred in zip(test['MatchID'], predictions):
    rows.append({
    'subtaskID': 2,
    'datapointID': id,
    'answer': test_indexed.loc[id, 'SAS']
})
    rows.append({
    'subtaskID': 3,
    'datapointID': id,
    'answer': pred.astype(int)
})
subDf = pd.DataFrame(rows)
subDf.to_csv('submission.csv', index=False)